In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

In [ ]:
# Directories
CODE_DIR = Path(r"C:\Users\willi\.vscode\Github\ml-from-crowd")
DATA_DIR = Path(r"E:\Research_data\Stocktwits\dataset\v1\data\csv")
FIGURES_DIR = Path(r"C:\Users\willi\.vscode\Github\Figures")
MODEL_DATA_DIR = Path(r"C:\Users\willi\.vscode\Github\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Sample period
SAMPLE_START = '2012-01-01'
SAMPLE_END = '2022-12-31'

def find_all_features_file(model_type):
    """Resolve the 'all features' prediction filename for a model type by finding
    the largest input-count file on disk (excludes the 2-feature baseline), so this
    doesn't need updating whenever the feature set changes."""
    candidates = [
        p for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl")
        if int(p.stem.split('input=')[1]) != 2
    ]
    if not candidates:
        return f"predictions_{model_type}_input=NOT_FOUND.pkl"
    return max(candidates, key=lambda p: int(p.stem.split('input=')[1])).name

# Model registry
MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_all': find_all_features_file('linear_regression'),
}

# Prediction target
TARGET = 'f_cumret1'

# Load and merge data

In [ ]:
# Load base data
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()

# Merge predictions from each model
for model_name, pred_file in MODELS.items():
    preds = pd.read_pickle(MODEL_DATA_DIR / pred_file)
    df[model_name] = preds.set_index('index')['prediction']

# Keep only rows with at least one prediction (OOS period)
model_cols = list(MODELS.keys())
df = df.dropna(subset=model_cols, how='all').reset_index(drop=True)

# Restrict to sample period
df['date'] = pd.to_datetime(df['date'])
df = df[(df['date'] >= SAMPLE_START) & (df['date'] <= SAMPLE_END)].reset_index(drop=True)
df['year_month'] = df['date'].dt.to_period('M')
df['year'] = df['date'].dt.year

print(f"OOS sample: {len(df):,} rows")
print(f"Period: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Models loaded: {model_cols}")
for col in model_cols:
    print(f"  {col}: {df[col].notna().sum():,} predictions")

# Daily cross-sectional rank correlations

In [ ]:
def daily_rank_corr(group, model_cols, target):
    """Compute Spearman rank correlation between each model's predictions and realized returns for one day."""
    result = {}
    for col in model_cols:
        valid = group[[target, col]].dropna()
        if len(valid) < 10:
            result[col] = np.nan
        else:
            corr, _ = spearmanr(valid[col], valid[target])
            result[col] = corr
    result['N'] = len(group)
    return pd.Series(result)

In [ ]:
daily_corr = df.groupby('date').apply(daily_rank_corr, model_cols=model_cols, target=TARGET)
daily_corr['year'] = daily_corr.index.year
daily_corr['year_month'] = daily_corr.index.to_period('M')

print(f"Daily rank correlations computed for {len(daily_corr):,} days")
print(f"\nSummary statistics:")
print(daily_corr[model_cols].describe())

# Full sample rank correlation

In [ ]:
# Full sample: average of daily cross-sectional rank correlations
full_sample = daily_corr[model_cols].mean()

print("Full Sample Average Daily Rank Correlation (Spearman)")
print("=" * 50)
for col in model_cols:
    print(f"  {col}: {full_sample[col]:.6f}")

# Yearly rank correlation

In [ ]:
# Yearly: average of daily cross-sectional rank correlations within each year
yearly_corr = daily_corr.groupby('year')[model_cols].mean()

print("Yearly Average Daily Rank Correlation (Spearman)")
print("=" * 50)
print(yearly_corr.to_string(float_format='{:.6f}'.format))

# Monthly rank correlation

In [ ]:
# Monthly: average of daily cross-sectional rank correlations within each month
monthly_corr = daily_corr.groupby('year_month')[model_cols].mean()

print("Monthly Average Daily Rank Correlation (Spearman)")
print("=" * 50)
print(monthly_corr.to_string(float_format='{:.6f}'.format))

# Plots

In [ ]:
# Daily rank correlation time series
fig, ax = plt.subplots(figsize=(14, 5))
for col in model_cols:
    ax.plot(daily_corr.index, daily_corr[col], label=col, alpha=0.4, linewidth=0.5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Daily Cross-Sectional Rank Correlation: Predictions vs Realized Returns')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly average rank correlation
fig, ax = plt.subplots(figsize=(14, 5))
x = monthly_corr.index.to_timestamp()
for col in model_cols:
    ax.plot(x, monthly_corr[col], label=col, marker='.', markersize=3)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Monthly Average Rank Correlation: Predictions vs Realized Returns')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly average rank correlation
fig, ax = plt.subplots(figsize=(10, 5))
x = yearly_corr.index.astype(int)
for col in model_cols:
    ax.plot(x, yearly_corr[col], marker='o', label=col)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Spearman Rank Correlation')
ax.set_title('Yearly Average Rank Correlation: Predictions vs Realized Returns')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily rank correlations
fig, axes = plt.subplots(1, len(model_cols), figsize=(7 * len(model_cols), 5))
if len(model_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, model_cols):
    vals = daily_corr[col].dropna()
    ax.hist(vals, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(x=vals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {vals.mean():.4f}')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Spearman Rank Correlation')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of Daily Rank Correlations: {col}')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()